# 02 — Macro Regime Analysis
Does the macro regime (hiking vs cutting rates, high vs low inflation)
actually matter for which JSE stocks perform well?

We use the regime-enriched master frame (`master_regimes_*.parquet`)
to test this empirically: regime duration, conditional forward returns,
momentum efficacy by regime, and a practical "best stocks for the
current regime" lookup.

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import sys

sys.path.insert(0, str(Path("..").resolve() / "src"))
from jse_radar.config import PROC_MASTER_DIR, PROC_MACRO_DIR

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)
print("Imports OK")

Imports OK


In [4]:
master_files = sorted(PROC_MASTER_DIR.glob("master_regimes_*.parquet"))
if not master_files:
    raise FileNotFoundError(
        "No master_regimes parquet found. Run MacroRegimeClassifier first."
    )

df = pd.read_parquet(master_files[-1])
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Regimes present: {df['composite_regime'].dropna().unique().tolist()}")

Shape: (79782, 57)
Date range: 2015-01-01 → 2026-06-26
Regimes present: ['HIKING_HIGH_INFLATION', 'CUTTING_HIGH_INFLATION', 'HIKING_TARGET_INFLATION', 'CUTTING_TARGET_INFLATION', 'CUTTING_LOW_INFLATION', 'HIKING_LOW_INFLATION']


## 1. Regime duration and frequency
How much calendar time has each regime occupied, and how persistent
is each one once it starts? A regime that lasts 2 months gives very
little time to act on; one that lasts 18 months is a real environment
to position for.

In [5]:
# Load the macro regime table directly (one row per month, not per ticker-day)
macro_files = sorted(PROC_MACRO_DIR.glob("macro_regimes_*.parquet"))
regimes = pd.read_parquet(macro_files[-1])
regimes["date"] = pd.to_datetime(regimes["date"])
regimes = regimes.dropna(subset=["composite_regime"]).sort_values("date")

# Count total months per regime
month_counts = regimes["composite_regime"].value_counts()

# Find each contiguous "spell" of a regime and its length
regimes["regime_changed"] = (
    regimes["composite_regime"] != regimes["composite_regime"].shift(1)
).cumsum()
spells = (
    regimes.groupby("regime_changed")
    .agg(
        regime=("composite_regime", "first"),
        start=("date", "first"),
        end=("date", "last"),
        months=("date", "count"),
    )
    .reset_index(drop=True)
)

avg_spell_length = spells.groupby("regime")["months"].mean().round(1)

summary = pd.DataFrame({
    "total_months": month_counts,
    "avg_spell_length_months": avg_spell_length,
    "num_spells": spells.groupby("regime").size(),
}).sort_values("total_months", ascending=False)

print(summary.to_string())

fig = px.bar(
    summary.reset_index().rename(columns={"index": "regime"}),
    x="regime", y="total_months",
    color="avg_spell_length_months",
    color_continuous_scale="Viridis",
    title="Total Months in Each Macro Regime (colour = avg spell length)",
    labels={"total_months": "Total Months", "regime": "Regime"},
    template="plotly_dark",
)
fig.update_layout(height=450, xaxis_tickangle=-30)
fig.show()

                          total_months  avg_spell_length_months  num_spells
CUTTING_TARGET_INFLATION            39                   2.6000          15
HIKING_TARGET_INFLATION             32                   2.5000          13
HIKING_HIGH_INFLATION               27                   6.8000           4
CUTTING_LOW_INFLATION                9                   2.2000           4
CUTTING_HIGH_INFLATION               4                   2.0000           2
HIKING_LOW_INFLATION                 1                   1.0000           1


## 2. Forward returns by regime
For each stock, what was its average forward 21-trading-day (≈1 month)
return when it was trading *during* each regime? This tells us whether
the regime classification carries real predictive signal, not just
descriptive colour.

We use **forward** returns (the return over the next 21 days from each
date) rather than the return that already happened, since what we care
about is: "if I knew the regime today, what return could I have expected
next?"

In [6]:
# Forward 21-day return per ticker: how much did the stock return
# over the NEXT month from this date?
df["fwd_return_21d"] = (
    df.groupby("ticker")["close"]
    .transform(lambda s: s.shift(-21) / s - 1)
)

# Drop rows with no regime or no forward return available
analysis_df = df.dropna(subset=["composite_regime", "fwd_return_21d"]).copy()

print(f"Usable rows for regime analysis: {len(analysis_df):,}")
print(f"\nObservations per regime:")
print(analysis_df["composite_regime"].value_counts().to_string())

Usable rows for regime analysis: 64,767

Observations per regime:
composite_regime
CUTTING_TARGET_INFLATION    22486
HIKING_TARGET_INFLATION     18456
HIKING_HIGH_INFLATION       15758
CUTTING_LOW_INFLATION        5264
CUTTING_HIGH_INFLATION       2187
HIKING_LOW_INFLATION          616


In [11]:
# Minimum number of historical observations required before we trust
# a ticker-regime average enough to show it. Below this threshold,
# a handful of unusual days can produce an extreme "average" that
# looks like signal but is really noise. We show these as blank
# (NaN) rather than letting them dominate the colour scale.
MIN_OBS = 20

# Compute both the average return AND the observation count per cell
grouped = analysis_df.groupby(["ticker", "composite_regime"])["fwd_return_21d"]
avg_return = grouped.mean().unstack() * 100
obs_count  = grouped.count().unstack()

# Mask out any cell with too few observations
pivot = avg_return.where(obs_count >= MIN_OBS)

# Order regimes logically
regime_order = [
    "HIKING_HIGH_INFLATION", "HIKING_TARGET_INFLATION", "HIKING_LOW_INFLATION",
    "CUTTING_HIGH_INFLATION", "CUTTING_TARGET_INFLATION", "CUTTING_LOW_INFLATION",
]
regime_order = [r for r in regime_order if r in pivot.columns]
pivot = pivot[regime_order]

# Sort tickers by their average return across all regimes (ignoring NaN)
pivot = pivot.loc[pivot.mean(axis=1, skipna=True).sort_values(ascending=False).index]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale="RdYlGn",
    zmid=0,
    text=pivot.values.round(2),
    texttemplate="%{text}",
    textfont={"size": 9},
    colorbar={"title": "Avg Fwd<br>21d Return (%)"},
    hoverongaps=False,
))

fig.update_layout(
    title=f"Average Forward 21-Day Return (%) by Ticker × Macro Regime<br>"
          f"<sup>Blank cells = fewer than {MIN_OBS} historical observations (not enough data to trust)</sup>",
    template="plotly_dark",
    height=700,
    xaxis_tickangle=-30,
)
fig.show()

In [14]:
# Two filters needed before we trust a ticker-regime cell:
#   1. MIN_OBS — enough total trading days (rules out tiny samples)
#   2. MIN_SPELLS — enough separate historical occurrences (rules out
#      "looks robust but is really just one lucky episode repeated
#      over many days")
MIN_OBS = 20
MIN_SPELLS = 2

# Count distinct spells per regime (reuse the `spells` table from Cell 5)
spell_counts = spells.groupby("regime").size()

# Regimes that don't meet the spell-count bar
unreliable_regimes = spell_counts[spell_counts < MIN_SPELLS].index.tolist()
print(f"Regimes with fewer than {MIN_SPELLS} independent spells "
      f"(treat with caution): {unreliable_regimes}")
print(spell_counts.sort_values().to_string())

grouped = analysis_df.groupby(["ticker", "composite_regime"])["fwd_return_21d"]
avg_return = grouped.mean().unstack() * 100
obs_count  = grouped.count().unstack()

# Mask out cells failing EITHER the observation count OR the spell count
pivot = avg_return.where(obs_count >= MIN_OBS)
for regime in unreliable_regimes:
    if regime in pivot.columns:
        pivot[regime] = np.nan

regime_order = [
    "HIKING_HIGH_INFLATION", "HIKING_TARGET_INFLATION", "HIKING_LOW_INFLATION",
    "CUTTING_HIGH_INFLATION", "CUTTING_TARGET_INFLATION", "CUTTING_LOW_INFLATION",
]
regime_order = [r for r in regime_order if r in pivot.columns]
pivot = pivot[regime_order]
pivot = pivot.loc[pivot.mean(axis=1, skipna=True).sort_values(ascending=False).index]

# Build column labels that flag unreliable regimes directly in the chart
col_labels = [
    f"{r} ⚠️" if r in unreliable_regimes else r
    for r in regime_order
]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=col_labels,
    y=pivot.index.tolist(),
    colorscale="RdYlGn",
    zmid=0,
    text=pivot.values.round(2),
    texttemplate="%{text}",
    textfont={"size": 9},
    colorbar={"title": "Avg Fwd<br>21d Return (%)"},
    hoverongaps=False,
))

fig.update_layout(
    title=f"Average Forward 21-Day Return (%) by Ticker × Macro Regime<br>"
          f"<sup>Blank = fewer than {MIN_OBS} observations. "
          f"⚠️ = fewer than {MIN_SPELLS} independent historical spells — treat with caution even where filled in</sup>",
    template="plotly_dark",
    height=700,
    xaxis_tickangle=-30,
)
fig.show()

Regimes with fewer than 2 independent spells (treat with caution): ['HIKING_LOW_INFLATION']
regime
HIKING_LOW_INFLATION         1
CUTTING_HIGH_INFLATION       2
HIKING_HIGH_INFLATION        4
CUTTING_LOW_INFLATION        4
HIKING_TARGET_INFLATION     13
CUTTING_TARGET_INFLATION    15


## 3. Does momentum work differently across regimes?
Momentum (stocks that have risen tend to keep rising) is a well
documented effect, but it's not constant — it can break down in
certain macro environments. We test this by splitting our existing
`momentum_score` signal by regime and checking its correlation with
forward returns in each one.

A positive correlation means: "when momentum_score was high, the
stock tended to keep rising" — i.e. momentum worked. Near zero or
negative means momentum was not a useful signal in that regime.

In [8]:
momentum_by_regime = (
    analysis_df.dropna(subset=["momentum_score"])
    .groupby("composite_regime")
    .apply(lambda g: g["momentum_score"].corr(g["fwd_return_21d"]))
    .reindex(regime_order)
    .dropna()
)

fig = px.bar(
    momentum_by_regime.reset_index(name="correlation"),
    x="composite_regime", y="correlation",
    color="correlation",
    color_continuous_scale="RdYlGn",
    title="Momentum Signal Efficacy by Regime<br>"
          "<sup>Correlation between momentum_score and forward 21-day return</sup>",
    labels={"correlation": "Correlation", "composite_regime": "Regime"},
    template="plotly_dark",
)
fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.4)
fig.update_layout(height=450, xaxis_tickangle=-30, coloraxis_showscale=False)
fig.show()

print(momentum_by_regime.round(4).to_string())

composite_regime
HIKING_HIGH_INFLATION      -0.0603
HIKING_TARGET_INFLATION     0.0490
HIKING_LOW_INFLATION        0.0607
CUTTING_HIGH_INFLATION     -0.1028
CUTTING_TARGET_INFLATION    0.0028
CUTTING_LOW_INFLATION      -0.0102


## 4. Regime hedges — consistent performers across hiking vs cutting
A genuinely useful stock for navigating uncertainty is one that
doesn't depend on guessing the regime correctly — it does reasonably
well in *both* hiking and cutting environments. We compute each
stock's average forward return in hiking regimes vs cutting regimes
and look for the smallest gap with positive returns in both.

In [9]:
analysis_df["rate_regime"] = analysis_df["composite_regime"].str.split("_").str[0]

hedge = (
    analysis_df.groupby(["ticker", "rate_regime"])["fwd_return_21d"]
    .mean()
    .unstack()
    * 100
)
hedge = hedge.dropna()
hedge["gap"] = (hedge["HIKING"] - hedge["CUTTING"]).abs()
hedge["both_positive"] = (hedge["HIKING"] > 0) & (hedge["CUTTING"] > 0)
hedge = hedge.sort_values("gap")

fig = px.scatter(
    hedge.reset_index(),
    x="HIKING", y="CUTTING",
    color="both_positive",
    text="ticker",
    title="Avg Forward Return: Hiking Regime vs Cutting Regime<br>"
          "<sup>Stocks near the diagonal perform similarly regardless of rate direction</sup>",
    labels={"HIKING": "Avg Fwd Return — Hiking (%)",
            "CUTTING": "Avg Fwd Return — Cutting (%)"},
    template="plotly_dark",
    color_discrete_map={True: "#3fb950", False: "#f85149"},
)
fig.add_hline(y=0, line_dash="dash", opacity=0.3)
fig.add_vline(x=0, line_dash="dash", opacity=0.3)
fig.update_traces(textposition="top center")
fig.update_layout(height=550)
fig.show()

print("Most consistent performers (smallest hiking/cutting gap, both positive):")
print(hedge[hedge["both_positive"]].head(8).to_string())

Most consistent performers (smallest hiking/cutting gap, both positive):
rate_regime  CUTTING  HIKING    gap  both_positive
ticker                                            
BTI.JO        0.3139  0.1749 0.1390           True
MRP.JO        0.8814  0.6987 0.1827           True
AGL.JO        2.1333  2.3339 0.2006           True
TBS.JO        0.2740  0.5201 0.2461           True
DSY.JO        1.0638  0.7955 0.2683           True
IMP.JO        2.8555  2.5724 0.2832           True
MTN.JO        0.4131  0.7334 0.3203           True
^J203.JO      0.8970  0.5627 0.3343           True


## 5. Practical takeaway — best stocks for the current regime
Given the most recent regime classification, which stocks have
historically performed best in exactly that regime? This is the
forward-looking, actionable output of the whole analysis.

In [10]:
current_regime = df.dropna(subset=["composite_regime"])["composite_regime"].iloc[-1]
print(f"Current macro regime: {current_regime}\n")

current_regime_performance = (
    analysis_df[analysis_df["composite_regime"] == current_regime]
    .groupby("ticker")
    .agg(
        avg_fwd_return_pct=("fwd_return_21d", lambda x: x.mean() * 100),
        win_rate_pct=("fwd_return_21d", lambda x: (x > 0).mean() * 100),
        observations=("fwd_return_21d", "count"),
    )
    .sort_values("avg_fwd_return_pct", ascending=False)
)

# Only show tickers with a reasonable number of historical observations
current_regime_performance = current_regime_performance[
    current_regime_performance["observations"] >= 10
]

fig = px.bar(
    current_regime_performance.reset_index().head(15),
    x="ticker", y="avg_fwd_return_pct",
    color="win_rate_pct",
    color_continuous_scale="Viridis",
    title=f"Top Historical Performers in Current Regime: {current_regime}<br>"
          "<sup>Color = % of historical occurrences with a positive forward return</sup>",
    labels={"avg_fwd_return_pct": "Avg Fwd 21d Return (%)", "ticker": "Stock"},
    template="plotly_dark",
)
fig.update_layout(height=500)
fig.show()

print(current_regime_performance.head(15).round(2).to_string())

Current macro regime: HIKING_LOW_INFLATION



          avg_fwd_return_pct  win_rate_pct  observations
ticker                                                  
IMP.JO               22.4700      100.0000            22
SOL.JO               21.0700       77.2700            22
GFI.JO                9.7800       90.9100            22
MTN.JO                8.8800      100.0000            22
BTI.JO                8.2700      100.0000            22
PRX.JO                5.1500      100.0000            22
TBS.JO                4.2100       77.2700            22
NPN.JO                3.8400       86.3600            22
DSY.JO                3.4000       81.8200            22
^J400.JO              3.3800      100.0000            22
^J203.JO              3.2000      100.0000            22
ABG.JO                3.0900      100.0000            22
SLM.JO                2.4900       81.8200            22
GRT.JO                1.7900       90.9100            22
AGL.JO                1.6400       54.5500            22


## Conclusion

This analysis set out to test whether macro regime classification carries
genuine predictive signal for JSE equity returns. The headline finding is
a methodological one as much as an empirical one: **of the six macro
regimes identified since 2015, only three have occurred often enough,
across enough independent historical episodes, to support reliable
conclusions** — `HIKING_HIGH_INFLATION`, `HIKING_TARGET_INFLATION`, and
`CUTTING_TARGET_INFLATION`.

The remaining three regimes (`HIKING_LOW_INFLATION`, `CUTTING_HIGH_INFLATION`,
`CUTTING_LOW_INFLATION`) initially appeared to show striking patterns —
in one case, average forward returns exceeding 50% for several tickers.
Applying a minimum-spell filter (requiring at least two independent
historical occurrences, not just a sufficient day count) revealed these
results were artifacts of one or two short, clustered episodes rather
than a repeatable regime effect. We treat these as **inconclusive** rather
than discard them outright, since South Africa's rate-cutting and
low-inflation episodes have simply been too rare and too brief in this
sample window to test properly — this is a data limitation, not evidence
that the regime effect doesn't exist.

Within the three well-sampled regimes, the dispersion in forward returns
across tickers was modest (single-digit percentages), which is the
expected result for a basket of large, liquid JSE names over a one-month
horizon — there is no standout regime-driven outperformer once
insufficiently-sampled regimes are excluded.

**Practical implication:** regime classification, as built here, is best
used as descriptive context (what environment are we in, and how long has
it lasted) rather than as a standalone predictive signal for stock
selection. A more robust test would require either a longer history
(pre-2015 SARB data) or pooling South Africa's regimes with comparable
emerging-market cutting/hiking cycles elsewhere, to build up enough
independent episodes for the lower-frequency regimes.

**Next steps:** extend the FRED/macro pull back further where data
permits, and revisit this analysis periodically — every additional
hiking or cutting cycle adds one more independent spell to regimes that
are currently under-sampled, which will sharpen these conclusions over time.